In [1]:
import numpy as np
import pickle
import re
import spacy
from spacy.lang.en.stop_words import STOP_WORDS
import unicodedata
from sentence_transformers import models, SentenceTransformer
from xgboost import XGBClassifier

In [2]:
# Load NLP model
nlp = spacy.load("en_core_web_sm", enable=["lemmatizer", "attribute_ruler", "tagger"])

In [3]:
# Load Sentence Transormer Model
transformer = models.Transformer("sentence-transformers/all-MiniLM-L12-v2")
pooling = models.Pooling(transformer.get_word_embedding_dimension(), pooling_mode="mean")
embedding_model = SentenceTransformer(modules=[transformer, pooling])

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/352 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [4]:
def clean_preprocess(text):
    doc = nlp(text)
    tokens = [
        token.lemma_.lower() for token in doc if
        token.is_ascii and
        not token.is_stop and
        not token.is_punct and
        not token.like_email and
        not token.is_space
    ]
    return " ".join(tokens)

In [12]:
def get_vector(cleaned_text):
    embedding = embedding_model.encode(cleaned_text)

    doc = nlp(str(cleaned_text))
    counts = {"PERSON": 0, "ORG": 0, "GPE": 0, "DATE": 0}
    for ent in doc.ents:
        if ent.label_ in counts:
            counts[ent.label_] += 1
    ner_array = np.array(list(counts.values()))

    return np.concatenate((embedding, ner_array))

In [13]:
def apply_model(text, classification_model):
    cleaned_text = clean_preprocess(text)
    vector = get_vector(cleaned_text)
    return classification_model.predict(vector.reshape(1, -1))

In [14]:
# Test Log Reg model
with open("models/RAG-Log-Reg.pkl", "rb") as f:
    loaded_model = pickle.load(f)

    response = "temp"
    while response:
        response = input("Enter a test prompt: ")
        if response == "q":
            break

        if apply_model(text=response, classification_model=loaded_model) == 1:
            print("External document retrieval is necessary")
        else:
            print("External document retrieval is not necessary")

External document retrieval is necessary


In [15]:
# Test XBoost model
with open("models/RAG-XGBoost.pkl", "rb") as f:
    loaded_model = pickle.load(f)
    
    response = "temp"
    while response:
        response = input("Enter a test prompt: ")
        if response == "q":
            break


        if apply_model(text=response, classification_model=loaded_model) == 1:
            print(f"External document retrieval is necessary to answer: {response}")
        else:
            print(f"External document retrieval is not necessary to answer: {response}")


External document retrieval is not necessary to answer: Who is Barack Obama?
External document retrieval is necessary to answer: What is the meaning of life?
External document retrieval is necessary to answer: What is the definition of 'stern'?
External document retrieval is not necessary to answer: Supercalifragilisticexpialidocious
External document retrieval is not necessary to answer: 
